In [ ]:
!pip install timm albumentations flask

In [ ]:
import os
from glob import glob
import json

model_dir = '/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/similar_fashion_products/train_results'
annotations_path = os.path.join(model_dir, 'annotations.json')
with open(annotations_path) as f:
    annotations = json.load(f)

print(f"이미지 개수: {len(annotations)}")

In [ ]:
annotations[:2]

In [ ]:
!FLASK_ENV=development FLASK_APP=/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/similar_fashion_products/part3_chapter03_app/app.py flask run

In [ ]:
import subprocess

flask_command = ["python3", "/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/similar_fashion_products/part3_chapter03_app/app.py"]

flask_process = subprocess.Popen(flask_command)

In [ ]:
import requests
import io
from PIL import Image
import random
from tqdm import tqdm
import numpy as np

feature_map = {}
margin = 0.1
random.shuffle(annotations)

for annot in tqdm(annotations[:10]):
    image_path = annot['image_path']
    image_id = annot['image_id']
    image = Image.open(image_path).convert('RGB')
    box = annot['box']

    w = box[2] - box[0]
    h = box[3] - box[1]

    new_box = [
        int(box[0] - w * margin),
        int(box[1] - h * margin),
        int(box[2] + w * margin),
        int(box[3] + h * margin),
    ]
    cropped_image = image.crop(new_box)

    image_bytes = io.BytesIO()
    cropped_image.save(image_bytes, format='JPEG')

    files = {'file': ('test_image.jpg', image_bytes.getvalue())}

    resp = requests.post(url="http://localhost:5000/predict", files=files)

    result = resp.json()
    feature_map[f"{image_path}_{image_id}"] = result["feature"]


In [ ]:
flask_process.terminate()

In [ ]:
feature_map.keys()

In [ ]:
import numpy as np

feat = np.array(feature_map)